# 📦 Notebook 5: Batching and Aggregation

Instead of writing every event immediately, batch them together or aggregate counters to reduce write load.

## Learning Objectives

By the end of this notebook, you'll understand:
- Write batching at different layers
- Counter aggregation patterns
- Hierarchical aggregation for fan-in
- Trade-offs of batching

In [ ]:
import psycopg2
import redis
import time
import random
from datetime import datetime
from collections import defaultdict
from typing import Dict, List

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "writes_demo",
    "user": "demo",
    "password": "demo",
}

def get_connection():
    c = psycopg2.connect(**DB_CONFIG)
    c.autocommit = True
    return c

# Seeded so the like distributions printed below are reproducible.
random.seed(42)

conn = get_connection()
r = redis.Redis(host="localhost", port=6379, decode_responses=True)
r.ping()

print("✅ Connected to PostgreSQL and Redis!")
print("📊 Open Adminer:      http://localhost:8080")
print("📊 Open RedisInsight: http://localhost:5540")


## 📦 Batching Layers

In [ ]:
print("📦 Where to Batch Writes")
print("=" * 60)
print("""
                    Batching Layers
─────────────────────────────────────────────────────────────

┌─────────────────────────────────────────────────────────┐
│                     APPLICATION                          │
│   • Buffer writes in memory                             │
│   • Flush every N items or T seconds                    │
│   • Risk: Data loss on crash                            │
└───────────────────────┬─────────────────────────────────┘
                        ▼
┌─────────────────────────────────────────────────────────┐
│                  INTERMEDIATE LAYER                      │
│   • Redis/Kafka as write buffer                         │
│   • Durable, can survive app crashes                    │
│   • Background workers batch to DB                      │
└───────────────────────┬─────────────────────────────────┘
                        ▼
┌─────────────────────────────────────────────────────────┐
│                      DATABASE                            │
│   • COPY command (PostgreSQL)                           │
│   • Bulk insert APIs                                    │
│   • Write-ahead log batching                            │
└─────────────────────────────────────────────────────────┘
""")

In [ ]:
class ApplicationBatcher:
    def __init__(self, batch_size: int = 100, flush_interval: float = 1.0):
        self.batch_size = batch_size
        self.flush_interval = flush_interval
        self.buffer: List[dict] = []
        self.last_flush = time.time()
        self.total_writes = 0
        self.total_flushes = 0
    
    def add(self, item: dict):
        self.buffer.append(item)
        self.total_writes += 1
        
        if len(self.buffer) >= self.batch_size:
            self.flush()
        elif time.time() - self.last_flush > self.flush_interval:
            self.flush()
    
    def flush(self):
        if not self.buffer:
            return
        
        self.total_flushes += 1
        batch_size = len(self.buffer)
        self.buffer = []
        self.last_flush = time.time()
        
        return batch_size
    
    def get_stats(self) -> dict:
        return {
            "total_writes": self.total_writes,
            "total_flushes": self.total_flushes,
            "db_write_reduction": f"{(1 - self.total_flushes/max(1, self.total_writes))*100:.1f}%"
        }

print("🔬 Application-Level Batching")
print("=" * 60)

batcher = ApplicationBatcher(batch_size=50)

print("\n📝 Writing 500 items with batch size 50...")
for i in range(500):
    batcher.add({"id": i, "value": f"data_{i}"})

batcher.flush()

stats = batcher.get_stats()
print(f"\n📊 Results:")
print(f"   Total writes: {stats['total_writes']}")
print(f"   Total flushes: {stats['total_flushes']}")
print(f"   DB write reduction: {stats['db_write_reduction']}")

print("\n✅ 500 writes → 10 database operations!")

# 500 items at batch_size=50 is exactly 10 flushes -- no more, no fewer. The
# final flush() is a no-op because the buffer emptied on the 500th add.
assert stats["total_writes"] == 500, stats
assert stats["total_flushes"] == 10, (
    f"500 items at batch_size=50 must be exactly 10 flushes, got {stats}"
)

print("\n⚠️  Note what this buffer does NOT survive: it is a plain Python list.")
print("   Kill the process between two flushes and up to 49 writes vanish with")
print("   no trace. That is the price of batching in the application layer, and")
print("   it is why the next layer down (Redis/Kafka) exists.")


## 👍 Counter Aggregation

In [ ]:
print("👍 The Like Counter Problem")
print("=" * 60)
print("""
PROBLEM: Viral post gets 10,000 likes per second
─────────────────────────────────────────────────────────────

NAIVE: Each like = 1 database write
    UPDATE posts SET likes = likes + 1 WHERE id = 123
    
    10,000 writes/second → Database explodes! 💥

─────────────────────────────────────────────────────────────

SOLUTION: Aggregate in Redis, periodically sync to DB

    ┌─────────┐      ┌─────────┐      ┌──────────┐
    │  Likes  │ ───> │  Redis  │ ───> │ Database │
    │ 10K/sec │      │ INCR    │      │ 1/minute │
    └─────────┘      │ counter │      └──────────┘
                     └─────────┘
    
    • Redis handles 100K+ INCR/sec
    • Database sees 1 write/minute per post
    • 600,000x write reduction!
""")

In [ ]:
class LikeAggregator:
    def __init__(self, conn):
        self.conn = conn
        self.prefix = "like_count"
        self.likes_received = 0
        self.syncs_performed = 0
    
    def add_like(self, post_id: int):
        key = f"{self.prefix}:{post_id}"
        r.incr(key)
        self.likes_received += 1
    
    def sync_to_db(self):
        cursor = self.conn.cursor()
        keys = r.keys(f"{self.prefix}:*")   # KEYS is O(N); use SCAN in production
        
        for key in keys:
            # Read the pending delta, apply it, and only THEN subtract it.
            #
            # GETDEL would be one call instead of two, but it removes the
            # counter *before* the UPDATE commits: crash in that window and
            # those likes are gone with nothing left to replay. Reading, then
            # committing, then DECRBY-ing the exact amount we applied makes the
            # flush at-least-once instead of at-most-once -- a crash re-applies
            # a delta (over-count, self-correcting on the next reconciliation)
            # rather than losing one (under-count, invisible forever).
            #
            # DECRBY rather than DEL also means a like that arrives *during*
            # the flush survives: it increments the key while we are working,
            # and we only subtract what we actually wrote.
            count = int(r.get(key) or 0)
            if count > 0:
                post_id = int(key.split(":")[1])
                cursor.execute(
                    "UPDATE post_metrics SET like_count = like_count + %s WHERE post_id = %s",
                    (count, post_id)
                )
                r.decrby(key, count)        # conn is autocommit -- UPDATE is durable here
                self.syncs_performed += 1
        
        cursor.close()

cursor = conn.cursor()
cursor.execute("DELETE FROM post_metrics")
for i in range(10):
    cursor.execute("INSERT INTO post_metrics (post_id, like_count) VALUES (%s, 0)", (i,))
cursor.close()

print("🔬 Like Aggregation Demo")
print("=" * 60)

aggregator = LikeAggregator(conn)

print("\n👍 Simulating 10,000 likes across 10 posts...")
import random
for _ in range(10000):
    post_id = random.randint(0, 9)
    aggregator.add_like(post_id)

print(f"   Likes received: {aggregator.likes_received}")
print(f"   Keys in Redis: {len(r.keys(f'{aggregator.prefix}:*'))}")

print("\n🔄 Syncing to database...")
aggregator.sync_to_db()

print(f"   Database writes: {aggregator.syncs_performed}")
print(f"   Write reduction: {(1 - aggregator.syncs_performed/aggregator.likes_received)*100:.2f}%")

cursor = conn.cursor()
cursor.execute("SELECT post_id, like_count FROM post_metrics ORDER BY like_count DESC LIMIT 5")
print("\n📊 Top 5 posts by likes:")
for row in cursor.fetchall():
    print(f"   Post {row[0]}: {row[1]} likes")
cursor.close()

print("\n✅ 10,000 likes → 10 database writes!")


def db_like_total() -> int:
    cur = conn.cursor()
    cur.execute("SELECT COALESCE(SUM(like_count), 0) FROM post_metrics")
    value = cur.fetchone()[0]
    cur.close()
    return value


# Aggregation is only acceptable if it is *lossless* in the happy path. A
# counter pattern that quietly drops increments is worse than no counter.
assert db_like_total() == aggregator.likes_received, (
    f"aggregation lost likes: {db_like_total()} in the DB vs "
    f"{aggregator.likes_received} received"
)
assert aggregator.syncs_performed == 10, (
    f"10 posts should cost exactly 10 UPDATEs, got {aggregator.syncs_performed}"
)

# A like that lands between two flushes must be picked up by the next one --
# the flush drains a delta, it does not reset a counter to zero.
r.incr(f"{aggregator.prefix}:0")
aggregator.likes_received += 1
aggregator.sync_to_db()
assert db_like_total() == aggregator.likes_received, (
    f"a like that arrived between flushes was dropped: {db_like_total()} vs "
    f"{aggregator.likes_received}"
)
print(f"   Flush is repeatable and conserving: DB total = {db_like_total():,} "
      f"after a second flush.")


## ⏱️ Is the Aggregator Actually Faster? Let's Measure.

The previous cell showed "10,000 likes -> 10 database writes" on paper.
But what does that mean for **wall-clock time and database load**?
Let's time the naive path (one `UPDATE` per like) against the
Redis-aggregated path (one `UPDATE` per post).


In [ ]:
print("⏱️ Naive UPDATE-per-like vs Aggregated-in-Redis")
print("=" * 60)

NUM_LIKES = 2000
NUM_POSTS = 10

cur = conn.cursor()
cur.execute("DELETE FROM post_metrics")
for i in range(NUM_POSTS):
    cur.execute(
        "INSERT INTO post_metrics (post_id, like_count) VALUES (%s, 0)", (i,)
    )
cur.close()
r.flushdb()

# 1) Naive: one UPDATE per like
start = time.time()
cur = conn.cursor()
for _ in range(NUM_LIKES):
    post_id = random.randint(0, NUM_POSTS - 1)
    cur.execute(
        "UPDATE post_metrics SET like_count = like_count + 1 WHERE post_id = %s",
        (post_id,),
    )
cur.close()
t_naive = time.time() - start
naive_db_writes = NUM_LIKES

# 2) Aggregated: INCR in Redis, then one UPDATE per post
cur = conn.cursor()
cur.execute("DELETE FROM post_metrics")
for i in range(NUM_POSTS):
    cur.execute(
        "INSERT INTO post_metrics (post_id, like_count) VALUES (%s, 0)", (i,)
    )
cur.close()

start = time.time()
for _ in range(NUM_LIKES):
    post_id = random.randint(0, NUM_POSTS - 1)
    r.incr(f"likes:{post_id}")

cur = conn.cursor()
agg_db_writes = 0
for post_id in range(NUM_POSTS):
    count = int(r.getdel(f"likes:{post_id}") or 0)
    if count:
        cur.execute(
            "UPDATE post_metrics SET like_count = like_count + %s WHERE post_id = %s",
            (count, post_id),
        )
        agg_db_writes += 1
cur.close()
t_agg = time.time() - start

print(f"\n1) Naive (1 UPDATE per like):")
print(f"   Time:      {t_naive:.3f}s")
print(f"   DB writes: {naive_db_writes}")
print(f"   Rate:      {NUM_LIKES/t_naive:,.0f} likes/sec")

print(f"\n2) Aggregated (Redis INCR + periodic flush):")
print(f"   Time:      {t_agg:.3f}s")
print(f"   DB writes: {agg_db_writes}")
print(f"   Rate:      {NUM_LIKES/t_agg:,.0f} likes/sec")

print(f"\n💡 Speedup: {t_naive/t_agg:.1f}x faster, "
      f"{naive_db_writes/max(1,agg_db_writes):.0f}x fewer DB writes.")
print("   Trade-off: like counts are briefly stale between flushes.")

# The point of the section is that collapsing N writes into one is both fewer
# DB writes AND less wall-clock time. If either half inverts, the "aggregate in
# Redis" advice printed above is no longer supported by this notebook.
assert agg_db_writes == NUM_POSTS, (
    f"aggregation should cost one UPDATE per post, got {agg_db_writes}"
)
assert naive_db_writes / agg_db_writes >= 100, (
    f"expected >=100x fewer DB writes, got "
    f"{naive_db_writes / agg_db_writes:.0f}x"
)
assert t_agg < t_naive, (
    f"expected the aggregated path to also be faster in wall-clock time, got "
    f"{t_agg:.3f}s vs {t_naive:.3f}s"
)


## ⚖️ The Half of Batching Nobody Benchmarks: Latency

The cell above measured the good news. Batching also has a bill, and the
exchange rate is not subtle: **you pay in freshness**.

If you flush every `T` seconds, a like that arrives just after a flush waits
almost the whole `T` before it is durable and visible in the database. Over a
steady arrival stream the mean staleness is `T/2` and the worst case is `T` —
those are not empirical constants, they fall straight out of the geometry.

Below we sweep the flush interval over a fixed arrival rate and measure both
sides at once. No `sleep` calls: this is a deterministic event-time simulation,
so the numbers are exact rather than a property of whatever else your laptop is
doing.

In [ ]:
print("⚖️ Write reduction vs staleness, as the flush interval grows")
print("=" * 60)

ARRIVAL_RATE = 2_000      # likes per second, steady
DURATION = 60.0           # seconds of traffic simulated
NUM_POSTS_SIM = 100       # distinct posts being liked

# Event-time simulation. Like i arrives at t = (i + 0.5) / rate -- the half-slot
# offset just keeps arrivals off the flush boundaries so nothing lands on a
# floating-point knife edge. A flush at time F makes durable everything that
# arrived before F.
arrivals = [(i + 0.5) / ARRIVAL_RATE for i in range(int(ARRIVAL_RATE * DURATION))]

print(f"\n{ARRIVAL_RATE:,} likes/sec across {NUM_POSTS_SIM} posts, "
      f"{DURATION:.0f}s of traffic\n")
print(f"{'flush every':>12} {'DB writes/min':>15} {'mean stale':>13} {'p99 stale':>13}")
print("-" * 56)

sweep = []
for interval in (0.0001, 0.01, 0.1, 1.0, 10.0, 60.0):
    staleness = []
    for t in arrivals:
        flush_at = (int(t / interval) + 1) * interval   # next flush boundary
        staleness.append(flush_at - t)
    staleness.sort()
    mean_stale = sum(staleness) / len(staleness)
    p99_stale = staleness[int(0.99 * len(staleness)) - 1]

    # Each flush writes one row per post that changed in the window -- capped
    # by how many posts exist. That cap is where batching starts to pay.
    writes_per_window = min(NUM_POSTS_SIM, ARRIVAL_RATE * interval)
    db_writes_per_min = (60.0 / interval) * writes_per_window

    sweep.append((interval, db_writes_per_min, mean_stale, p99_stale))
    print(f"{interval:>11.4f}s {db_writes_per_min:>15,.0f} "
          f"{mean_stale * 1000:>11.1f}ms {p99_stale * 1000:>11.1f}ms")

saturation = NUM_POSTS_SIM / ARRIVAL_RATE
print(f"""
💡 The two columns have to be read against each other:

   • Below T = {saturation * 1000:.0f} ms a window holds fewer likes than there are posts, so
     almost every like still costs its own row. Batching buys you nothing but
     latency down here. Batching only pays once writes COLLIDE on the same key
     -- which is exactly why it is a counter/aggregation technique and not a
     general-purpose speedup.
   • Above that point, DB writes fall linearly with T while staleness grows
     linearly with T. Going from T=0.1s to T=1s cuts writes 10x and costs
     450 ms of extra mean staleness. There is no knee to find, no free lunch
     left -- it is a straight-line trade you have to price.

   So choose T from the product requirement ("like counts may be up to 2s
   stale"), never from the throughput number. Throughput follows.
""")

# Mean staleness of a uniform arrival stream flushed every T is T/2, and no
# item can ever wait longer than T. If either stops holding, this simulation
# has drifted away from the thing it claims to model.
for interval, _writes, mean_stale, p99_stale in sweep:
    assert abs(mean_stale - interval / 2) < interval * 0.02, (
        f"mean staleness at T={interval}s should be ~T/2={interval / 2:.4f}s, "
        f"got {mean_stale:.4f}s"
    )
    assert p99_stale <= interval + 1e-9, (
        f"staleness can never exceed the flush interval, got "
        f"{p99_stale:.4f}s > {interval}s"
    )

# And the whole point: a longer interval must never cost MORE database writes.
write_counts = [w for _i, w, _m, _p in sweep]
assert write_counts == sorted(write_counts, reverse=True), (
    f"longer flush intervals must mean fewer (or equal) DB writes, got "
    f"{write_counts}"
)
# The far ends of the sweep have to differ by orders of magnitude, otherwise
# the table is not showing a trade-off at all.
assert write_counts[0] / write_counts[-1] > 100, (
    f"expected the sweep to span >100x in write volume, got "
    f"{write_counts[0] / write_counts[-1]:.0f}x"
)


## 🌳 Hierarchical Aggregation

In [ ]:
print("🌳 Hierarchical Aggregation")
print("=" * 60)
print("""
PROBLEM: 1000 servers all writing to 1 database
─────────────────────────────────────────────────────────────

    Server 1 ──┐
    Server 2 ──┤
    Server 3 ──┼───> [Single Database] 💥 Bottleneck!
       ...     │
    Server N ──┘

─────────────────────────────────────────────────────────────

SOLUTION: Hierarchical aggregation (fan-in)

    Level 0 (1000 servers):
    ┌───┐ ┌───┐ ┌───┐     ┌───┐
    │ S1│ │ S2│ │ S3│ ... │S1K│  Buffer locally
    └─┬─┘ └─┬─┘ └─┬─┘     └─┬─┘
      │     │     │         │
      └──┬──┘     └────┬────┘
    
    Level 1 (100 aggregators):
       ┌───────┐      ┌───────┐
       │ Agg 1 │      │ Agg N │   Batch 10 servers each
       └───┬───┘      └───┬───┘
           │              │
           └──────┬───────┘
    
    Level 2 (10 aggregators):
             ┌─────────┐
             │ Final   │
             │ Agg     │   Batch 10 aggregators each
             └────┬────┘
                  │
                  ▼
            [Database]  ← Only 10 writers!

1000 servers → 10 database connections
""")

In [ ]:
class HierarchicalAggregator:
    def __init__(self, level: int, batch_threshold: int = 10):
        self.level = level
        self.batch_threshold = batch_threshold
        self.buffer: Dict[str, int] = defaultdict(int)
        self.upstream: 'HierarchicalAggregator' = None
        self.flushes = 0
        
    def set_upstream(self, upstream: 'HierarchicalAggregator'):
        self.upstream = upstream
    
    def write(self, key: str, value: int = 1):
        self.buffer[key] += value
        
        if sum(self.buffer.values()) >= self.batch_threshold:
            self.flush()
    
    def flush(self):
        if not self.buffer:
            return
            
        self.flushes += 1
        
        if self.upstream:
            for key, value in self.buffer.items():
                self.upstream.write(key, value)
        
        self.buffer.clear()

print("🔬 Hierarchical Aggregation Demo")
print("=" * 60)

db_writer = HierarchicalAggregator(level=2, batch_threshold=100)

mid_aggregators = [HierarchicalAggregator(level=1, batch_threshold=50) for _ in range(10)]
for agg in mid_aggregators:
    agg.set_upstream(db_writer)

servers = [HierarchicalAggregator(level=0, batch_threshold=10) for _ in range(100)]
for i, server in enumerate(servers):
    server.set_upstream(mid_aggregators[i // 10])

print("\n📝 100 servers each writing 100 events...")
for server in servers:
    for _ in range(100):
        key = f"metric_{random.randint(0, 9)}"
        server.write(key)

for server in servers:
    server.flush()
for agg in mid_aggregators:
    agg.flush()
db_writer.flush()

server_flushes = sum(s.flushes for s in servers)
mid_flushes = sum(a.flushes for a in mid_aggregators)
db_flushes = db_writer.flushes

print(f"\n📊 Results:")
print(f"   Total events: 10,000")
print(f"   Level 0 (servers) flushes: {server_flushes}")
print(f"   Level 1 (mid-tier) flushes: {mid_flushes}")
print(f"   Level 2 (database) flushes: {db_flushes}")
print(f"   Write reduction: {(1 - db_flushes/10000)*100:.2f}%")

print("\n✅ 10,000 events → ~100 database writes!")

# Each level divides by its batch threshold: 10,000 events -> 10,000/10 = 1,000
# server flushes -> 10,000/50 = 200 mid-tier flushes -> 10,000/100 = 100 DB
# writes. Deterministic regardless of which metric key each event picks, since
# the thresholds count events rather than keys.
assert server_flushes == 1000, f"expected 1,000 level-0 flushes, got {server_flushes}"
assert mid_flushes == 200, f"expected 200 level-1 flushes, got {mid_flushes}"
assert db_flushes == 100, f"expected 100 level-2 flushes, got {db_flushes}"
assert db_flushes < mid_flushes < server_flushes, (
    "each level must fan in further than the one below it, got "
    f"{server_flushes}/{mid_flushes}/{db_flushes}"
)

print("\n⚠️  Honest accounting: the tree did not make writes disappear, it moved")
print("   them. 1,000 + 200 + 100 = 1,300 flush operations happened in total;")
print("   only 100 of them were the expensive ones (the database). You bought")
print("   that with two extra network hops of latency and two more tiers of")
print("   in-memory buffer that a crash can drop on the floor.")


## 🧪 Quick Quiz

1. **What's the trade-off of counter aggregation?**

2. **When would you use hierarchical aggregation?**

3. **What happens if an aggregator crashes before syncing?**

In [ ]:
print("📝 Quiz Answers")
print("=" * 50)
print()
print("1. Counter aggregation trade-off:")
print("   - Counts are eventually consistent")
print("   - May show slightly stale values")
print("   - Sync failure = lost increments")
print()
print("2. Use hierarchical aggregation when:")
print("   - Many sources writing to one destination")
print("   - Fan-in pattern (1000s → 1)")
print("   - Example: metrics from 1000 servers")
print()
print("3. Aggregator crash:")
print("   - In-memory buffer is lost")
print("   - Solution: Write-ahead log (WAL)")
print("   - Or use Redis as durable buffer")

## 📚 Summary

### Key Takeaways

1. **Batch at multiple layers** - App, middleware, database
2. **Counter aggregation** - Buffer in Redis, sync periodically
3. **Hierarchical aggregation** - For fan-in patterns
4. **Trade-offs** - Eventual consistency, crash recovery
5. **Massive reduction** - 10,000x fewer database writes possible

### Next Up

In **Notebook 6**, we'll learn about hot keys:
- Detecting hot keys
- Key splitting strategies
- Dynamic resharding